# 5 - IP Resample IFSAR Mosaics and Create Valley Bottom Polygons

Use this tool to resample 5m IFSAR mosaics to 100m resolution. The 100m DEM is then used to compute slope, which is used in establishing a cost surface for defining valley bottom polygons.

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.


## Required Inputs:

- A geodatabase containing NetMap stream reaches clipped to HUC8 polygon extents. These feature classes must have a "reach" prefix and "HUC8" suffix in the feature class name.

- A collection of associated mosaicked IFSAR 5m DEMs clipped to HUC8 polygon extents. **These must be loaded into the current ArcPro project map**.... These should be the output from the "IP 3 Merge IFSAR Tiles by HUC8 Extent" tool, and should have a "DEM_5m" prefix.  


## Geoprocessing Output:

- A collection of 100m resampled DEMs and slope rasters derived from them, written to the default project geodatabase.

## Processing Steps:

1. List the clipped 5m DEMs in the map project. 
2. Resample each 5m DEM to 100m resolution.
3. Compute slope rasters for each 100m DEM.
4. List reaches in the user-provided GDB, and sort them in the same order as the slope rasters.
5. Run each reach and slope raster combination through the valley bottom polygon processing workflow.

### Code starts here:

#### Setup

Import modules and reset environments to default. This should set the ArcPro project geodatabase as the workspace/scratch environment, just in case it was set otherwise. Additionally, prevent the addition of intermediary outputs to the ArcPro project map. Some tools may not run if their target is open in the map display.

In [85]:
import arcpy
arcpy.ResetEnvironments()
arcpy.env.addOutputsToMap = False

User provides filepaths to the geodatabase, the field in the HUC8 reach feature class that contains the unique reach ID (usually "RCA_ID" in IP toolbox script outputs), and the name of the current ArcPro map with the clipped 5m DEMs loaded into it.

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [86]:
gdb = "F:\GIS\IP\Yukon_Tanana.gdb"
rid = "RCA_ID"
map_name = "IP_BASE"

In [193]:
#gdb = arcpy.GetParameterAsText(0)
#rid = arcpy.GetParameterAsText(1)
#map_name = arcpy.GetParameterAsText(2)

#### map_name variable should be defined using the ArcPro toolbox parameter data type "GPMap" !!!

List the clipped 5m DEMs in the map project. They must have a prefix of "DEM_5m" for this to work. 

In [87]:
p = arcpy.mp.ArcGISProject("CURRENT")
m = p.listMaps(map_name)[0]

layers = []

for lyr in m.listLayers():
    l = lyr.name
    if l.startswith("DEM_5m") == True:
        layers.append(l)
        
layers

['DEM_5m_19080311_clip.tif', 'DEM_5m_19080310_clip.tif', 'DEM_5m_19080309_clip.tif', 'DEM_5m_19080308_clip.tif', 'DEM_5m_19080307_clip.tif', 'DEM_5m_19080306_clip.tif', 'DEM_5m_19080305_clip.tif', 'DEM_5m_19080304_clip.tif', 'DEM_5m_19080303_clip.tif', 'DEM_5m_19080302_clip.tif', 'DEM_5m_19080301_clip.tif']

Resample each 5m DEM to 100m resolution.

In [88]:
dems_100m = []

for dem in layers:
    
    desc = arcpy.Describe(dem)
    dempath = str(desc.path + "/" + dem)
    
    outname = str(dem.split("5m")[0] + "100m" + dem.split("5m")[1])
    outname = outname.split(".tif")[0]
    
    arcpy.management.Resample(dempath, outname, "100", "BILINEAR")
    
    dems_100m.append(outname)

dems_100m

['DEM_100m_19080311_clip', 'DEM_100m_19080310_clip', 'DEM_100m_19080309_clip', 'DEM_100m_19080308_clip', 'DEM_100m_19080307_clip', 'DEM_100m_19080306_clip', 'DEM_100m_19080305_clip', 'DEM_100m_19080304_clip', 'DEM_100m_19080303_clip', 'DEM_100m_19080302_clip', 'DEM_100m_19080301_clip']

Make and save slope rasters for each 100m DEM.

In [89]:
slps_100m = []

for d in dems_100m:
    slpname = str(d + "_slope")
    slp = arcpy.sa.Slope(d, "PERCENT_RISE")
    slp.save(slpname)
        
    desc = arcpy.Describe(slpname)
    slp_full = str(desc.path + "/" + slpname)

    slps_100m.append(slp_full)
    
slps_100m

['F:\\GIS\\IP\\IP.gdb/DEM_100m_19080311_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080310_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080309_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080308_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080307_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080306_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080305_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080304_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080303_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080302_clip_slope', 'F:\\GIS\\IP\\IP.gdb/DEM_100m_19080301_clip_slope']

List reaches in the user-provided GDB. Then order the list using the HUC codes that occur in both the reach name and in the slope name.

In [90]:
arcpy.env.workspace = gdb

featureclasses = arcpy.ListFeatureClasses()

reaches = []

for fc in featureclasses:
    if (fc.startswith("reach") == True) and (fc.endswith("HUC8") == True):
        
        desc = arcpy.Describe(fc)
        fc_full = str(desc.path + "/" + fc)
        
        reaches.append(fc_full)
        
reaches

['F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080301_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080302_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080305_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080309_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080308_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080310_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080304_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080306_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080311_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080307_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080303_HUC8']

In [91]:
reaches_sorted = []

for s in slps_100m:
    code = s.split("_")[-3]
    
    for r in reaches:

        r_code = r.split("_")[-2]
        
        if r_code == code:
            reaches_sorted.append(r)
        
        else:
            pass

reaches_sorted

['F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080311_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080310_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080309_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080308_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080307_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080306_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080305_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080304_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080303_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080302_HUC8', 'F:\\GIS\\IP\\Yukon_Tanana.gdb/reach_Tanana_5km_19080301_HUC8']

Now that they are sorted, we can use each slope and reach combination to create valley bottom polygons.

The following process converts the reaches to raster using the user-provided unique reach ID field to provide raster values. Then the distance accumulation function is run using the 100m slope raster as the cost surface, and 1.75 as the source cost multiplier. 

By visual inspection, a distance accumulation value of >= 600 seemed to capture valley bottom areas similar in size and shape to NetMap valley bottoms in the SERDP study area. (This can be adjusted if necessary for other study areas.) Use the value of 600 as a threshold to set all values <=600 as NULL, and all values >600 as 1 . 

Convert the non-null raster values to polygons and save the output in the user-defined geodatabase. Then delete all intermediary rasters.

In this way, there will be a separate valley bottom polygon feature class for each HUC8 reach feature class. These will later be clipped by RCA and cleaned of small polygons using subsequent scripts.

In [92]:
for s, r in zip(slps_100m, reaches_sorted):
    
    slope = arcpy.sa.Raster(s)
    
    arcpy.conversion.FeatureToRaster(r, rid, "edges_raster", 100)
    
    outDistAcc = arcpy.sa.DistanceAccumulation("edges_raster", "", "", slope, "", "", "", "", "", "", "", "", "", 1.75, "", "")

    DistAcc_600 = arcpy.sa.Con(outDistAcc, 1, -123, 'VALUE <= 600')

    DistAcc_600_Nulled = arcpy.sa.SetNull(DistAcc_600, DistAcc_600, 'VALUE = -123')
    
    arcpy.conversion.RasterToPolygon(DistAcc_600_Nulled, "VB600_", "SIMPLIFY", "", "", "")
    
    final_out = str( r+ "_VB600_smoothed")
    
    arcpy.cartography.SmoothPolygon("VB600_", final_out, "PAEK", 250)
    
    arcpy.management.Delete("edges_raster")
    arcpy.management.Delete("VB600_")
    arcpy.management.Delete(outDistAcc)
    arcpy.management.Delete(DistAcc_600)
    arcpy.management.Delete(DistAcc_600_Nulled)